In [1]:
# DDP AlexNet + ImageNet Example (Notebook)

# This notebook shows:
# 1. How to define a local training process with alignment + progressive dropout.
# 2. How to incorporate multi-GPU DDP by calling external script.

# **Important**: If you want to run fully in a notebook environment with DDP, you must spawn processes (not trivial inside notebooks). Usually, we do multi-GPU from a script. 

In [2]:
# Cell 1: Imports
%reload_ext autoreload
%autoreload 2

import torch
import os
import sys

# Insert your alignment_v2 path if needed:
# sys.path.insert(0, "/path/to/alignment_v2")

from alignment_v2.models.registry import get_model
from alignment_v2.datasets import get_dataset
from alignment_v2 import train
from alignment_v2.train import progressive_dropout, progressive_dropout_experiment
from alignment_v2.experiments.experiment import Experiment
from alignment_v2 import plotting
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on device:", DEVICE)

ImportError: cannot import name 'progressive_dropout_experiment' from 'alignment_v2.train' (/n/holylabs/LABS/kempner_dev/Users/hsafaai/Code/alignment/src/alignment_v2/train.py)

In [ ]:
# Cell 2: Build a single-GPU example (no DDP) for demonstration here.

model_name = "AlexNet"
dataset_name = "ImageNet"
learning_rate = 1e-3
weight_decay = 0
dropout_rate = 0
replicates = 1

# Build the model (single GPU)
nets = [
    get_model(
        model_name,
        build=True,
        dataset=dataset_name,
        dropout=dropout_rate,
        ignore_flag=False,
    ).to(DEVICE)
    for _ in range(replicates)
]

optimizers = [torch.optim.Adam(net.parameters(), lr=learning_rate, weight_decay=weight_decay) for net in nets]

# Build the dataset (single-GPU)
loader_params = dict(
    batch_size=64,
    shuffle=True,
    num_workers=2,
)
imagenet_dataset = get_dataset(dataset_name,
                               build=True,
                               transform_parameters=nets[0],  # alignment transforms
                               loader_parameters=loader_params,
                               device="cpu",  # do transforms on CPU
                               distributed=False)

# Now do a short training run to test logic:
train_params = dict(
    num_epochs=2,
    alignment=True,
    alignment_expansion=False,
    compare_expected=False,
    frequency=1,
    delta_alignment=False,
)

results = train.train(nets, optimizers, imagenet_dataset, **train_params)
print("Training done. Results keys:", list(results.keys()))

In [ ]:
# Cell 3: Progressive Dropout Example
dropout_params = {
    "num_drops": 5,
    "by_layer": False,
    "train_set": False,
    "retrain": False,
}

dropout_res = progressive_dropout(nets, imagenet_dataset, alignment=results["alignment"], **dropout_params)
print("Dropout results keys:", list(dropout_res.keys()))

# Plot
plotting.plot_dropout_results(exp=None, dropout_results=dropout_res,
                              dropout_parameters=dropout_params,
                              prms={
                                  "vals":[model_name],
                                  "name":"ModelType",
                                  "dataset":dataset_name,
                                  "dropout":dropout_rate,
                                  "lr":learning_rate,
                                  "weight_decay":weight_decay
                               },
                              dropout_type="alignment-based")
plt.show()